## LLAMA - budowanie własnego chatbota

**NOTE** Ten notebook zadziała tylko na środowisku z GPU T4.

In [ ]:
!pip install transformers accelerate bitsandbytes

### Załadowanie modelu LLaMA lokalnie

Załadowanie modelu **LLaMA** lokalnie może zająć kilka sekund.

Aby móc pracować z modelem LLaMA na własnym komputerze, wymagane jest konto w serwisie Hugging Face. Następnie należy uzyskać dostęp do wybranego modelu poprzez złożenie odpowiedniego wniosku.

---

### Jak uzyskać dostęp do modelu i token API

#### 1. Załóż konto
- Wejdź na: https://huggingface.co/
- Zarejestruj się (email / GitHub / Google)

#### 2. Znajdź model LLaMA
- Przejdź do strony modelu, np.:
  - `meta-llama/Llama-2-7b`
  - `meta-llama/Meta-Llama-3-8B`
- Modele od Meta wymagają akceptacji licencji.

#### 3. Poproś o dostęp
- Kliknij **“Request access”** na stronie modelu
- Zaakceptuj warunki licencji
- Czas oczekiwania: od kilku minut do kilku dni

#### 4. Wygeneruj token dostępu
- Przejdź do: *Settings → Access Tokens*
- Utwórz nowy token (najlepiej z uprawnieniami **Read**)
- Skopiuj token


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
model_name = "meta-llama/Llama-3.2-3B-Instruct" # "meta-llama/Llama-2-7b-chat-hf"
prompt = "Tell me a joke about artificial neural networks"
access_token = "PUT_YOUR_TOKEN_HERE"


tokenizer = AutoTokenizer.from_pretrained(model_name, token=access_token)
model = AutoModelForCausalLM.from_pretrained(model_name, token=access_token)

Test modelu

In [ ]:
model_inputs = tokenizer(prompt, return_tensors="pt")

output = model.generate(**model_inputs)

In [ ]:
print(output[0])

In [ ]:
print(tokenizer.decode(output[0], skip_special_tokens=True))

### Zadanie - Ekstrakcja encji (Named Entity Recognition) z użyciem modelu LLaMA

Zadaniem jest wykonanie **ekstrakcji encji (NER – Named Entity Recognition)** przy użyciu modelu LLaMA na podanym tekście.

Model powinien zidentyfikować i wyodrębnić następujące typy informacji, a następnie zwrócić je w formacie **JSON (klucz–wartość)**:

- `PERSON` – osoby (np. imiona i nazwiska),
- `ORGANIZATION` – organizacje (np. firmy, instytucje),
- `LOCATION` – lokalizacje (np. miasta, kraje),
- `DATE` – daty,
- `MONEY` – wartości pieniężne.

---

### Krótki przykład

#### Tekst wejściowy:

`Elon Musk visited Berlin in 2023 and invested 10 million dollars in a new Tesla factory.`


#### Oczekiwany wynik:
```json
{
  "PERSON": ["Elon Musk"],
  "ORGANIZATION": ["Tesla"],
  "LOCATION": ["Berlin"],
  "DATE": ["2023"],
  "MONEY": ["10000000"]
}
```

Zwykle rozwiązuje się to jako odpowiedni prompt/system prompt. Możliwe jest też fine-tunowanie transformerów do wybranych zadań.

In [ ]:
prompt_enity = """
On March 15, 2023, OpenAI announced a new partnership with Microsoft during a press conference held in San Francisco, California. The collaboration aims to integrate advanced AI models into Microsoft’s Azure cloud platform, with a projected investment of over $1 billion.
According to Sam Altman, CEO of OpenAI, the partnership will accelerate the democratization of artificial intelligence across industries. In a related development, Google DeepMind unveiled a new version of its Gemini model on the same day in London.
Meanwhile, the European Commission is preparing new AI regulations, expected to be finalized by Q4 of 2024. These regulations could affect companies like Amazon, Meta, and IBM, which are actively developing generative AI applications.
Analysts from Goldman Sachs and McKinsey & Company predict that AI could contribute up to $15.7 trillion to the global economy by 2030, with the healthcare and finance sectors likely to benefit the most.
"""


**TIP** For this inference a transformer pipeline might be more suitable - **restart kernel** before execution.

In [ ]:
import torch
from transformers import pipeline

In [ ]:
model_id = "meta-llama/Llama-3.2-3B-Instruct"
pipe = pipeline(
    "text-generation",
    model=model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

In [ ]:
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": prompt_entity},
]

In [ ]:
outputs = pipe(
    messages,
    max_new_tokens=256,
)
print(outputs[0]["generated_text"][-1])

In [ ]:
print(outputs[0]["generated_text"][-1]["content"])

### Zadanie 1 – Testowanie modelu LLaMA na zadaniach programistycznych

**ZADANIE**  
Przetestuj model LLaMA na kilku zadaniach programistycznych, używając własnych promptów:

- Napisz funkcję w Pythonie, która oblicza n-tą liczbę Fibonacciego rekurencyjnie.  
  Uwzględnij:
  - podstawową walidację wejścia,
  - docstring opisujący działanie funkcji.

- Wymyśl i przetestuj jedno własne zadanie programistyczne (dowolne).

---

**WSKAZÓWKA**  
Możesz przetestować modele dostrojone specjalnie do kodu (jeśli masz do nich dostęp):  
https://huggingface.co/meta-llama

---



In [ ]:
coding_prompt = """
PUT_YOUR_PROMPT_HERE
"""

Prosty chatbot z użyciem lokalnego modelu LLaMA

**ZADANIE DODATKOWE**  
Zbuduj prostą klasę chatbota wykorzystującą lokalny model LLaMA:

- Zdefiniuj bazowy prompt (*system prompt*), który określa specjalizację modelu (np. ekspert od programowania, matematyki itd.).
- Przekazuj pytanie użytkownika do modelu, doklejając je do bazowego promptu  
  (np. poprzez metodę `invoke`).
- Zwracaj odpowiedź modelu jako string (po przetworzeniu wyjścia modelu na tekst).

---

**Rozszerzenie ($)**

- Przechowuj historię ostatnich 3 interakcji (pytanie + odpowiedź).
- Dodawaj tę historię do promptu, aby model miał kontekst rozmowy.
- Jeśli historia staje się zbyt długa:
  - zamiast pełnych wiadomości użyj ich skróconych podsumowań,
  - dodaj te podsumowania do promptu zamiast pełnego tekstu.

---

### Cel

- Zrozumienie, jak działa prompt engineering w praktyce,
- Testowanie zdolności modelu do generowania kodu,
- Implementacja prostego systemu konwersacyjnego z pamięcią kontekstu.

Z poniższych materiałów można dowiedzieć się więcej o transformerach:
- Artykuł o mechaniźmie atencji (~ 175k cites): https://arxiv.org/abs/1706.03762
- https://www.youtube.com/watch?v=KJtZARuO3JY&ab_channel=GrantSanderson
